# AGN Variability: Gaia DR3 × Rubin DP2 DRW Analysis

Cross-match Gaia DR3 AGN candidates with Rubin DP2 difference-imaging lightcurves,
fit a damped random walk (DRW) model per band with EzTaoX, and identify the most
variable AGN on ~10-day scales.

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import lsdb
import eztaox
from dask.distributed import Client
from astroquery.gaia import Gaia

# Suppress astroquery verbose output
import logging
logging.getLogger('astroquery').setLevel(logging.WARNING)

BAND_COLORS = {
    'u': '#9B59B6', 'g': '#2ECC71', 'r': '#E67E22',
    'i': '#E74C3C', 'z': '#8B0000', 'y': '#4A235A'
}
BAND_ORDER  = ['u', 'g', 'r', 'i', 'z', 'y']
BAND_WL_UM  = {'u': 0.370, 'g': 0.480, 'r': 0.620, 'i': 0.760, 'z': 0.870, 'y': 0.970}

In [ ]:
client = Client(
    n_workers=8,
    threads_per_worker=1,
    memory_limit='8GB',
    local_directory='/tmp/dask-scratch',
)
print('Dashboard:', client.dashboard_link)

## 2. Load the Rubin DP2 DIA object collection

Opening as a collection automatically attaches the margin catalog.

In [ ]:
DIA_PATH = '/sdf/data/rubin/shared/lsdb_commissioning/hats/v30_0_6/dia_object_collection'

dia_objects = lsdb.open_catalog(DIA_PATH)

print('Columns:', list(dia_objects.columns))
print('Approx objects:', len(dia_objects))
if dia_objects.margin is not None:
    print('Margin threshold:', dia_objects.margin.hc_structure.catalog_info.margin_threshold, 'arcsec')
else:
    print('WARNING: no margin attached — crossmatch results near partition edges may be incomplete')

In [ ]:
# Inspect schema: identify the nested sources column
sample = dia_objects.head(3)
print(sample.dtypes)
print()

# The nested column is whichever dtype is nested_pandas.NestedDtype
# (often 'sources' or 'diaSource' depending on how the collection was built)
import nested_pandas as npd
nested_cols = [
    c for c in sample.columns
    if hasattr(sample[c].dtype, 'name') and 'nested' in str(sample[c].dtype).lower()
]
# Fallback: look for any column whose values are DataFrames
if not nested_cols:
    nested_cols = [
        c for c in sample.columns
        if isinstance(sample[c].iloc[0], pd.DataFrame)
    ]
print('Nested columns detected:', nested_cols)

# !! ADJUST THIS IF THE DETECTED NAME DIFFERS !!
NESTED_COL = nested_cols[0] if nested_cols else 'sources'

In [ ]:
# Show the sub-schema of the nested sources column
first_row = sample.iloc[0]
if NESTED_COL in first_row.index and first_row[NESTED_COL] is not None:
    sub = first_row[NESTED_COL]
    print(f'Nested column "{NESTED_COL}" sub-schema:')
    print(sub.dtypes)
    print(sub.head(2))

# Rubin DIA source column names used below — adjust if your schema differs
TIME_COL     = 'midpointMjdTai'   # observation epoch
FLUX_COL     = 'psfFlux'          # PSF flux (nJy)
FLUXERR_COL  = 'psfFluxErr'       # PSF flux uncertainty (nJy)
BAND_COL     = 'band'             # filter band: u/g/r/i/z/y

# Pixel-level quality flags to reject
FLAG_COLS = [
    'pixelFlags_saturated',
    'pixelFlags_saturatedCenter',
    'pixelFlags_cr',
    'pixelFlags_crCenter',
    'pixelFlags_bad',
    'pixelFlags_edge',
    'pixelFlags_noData',
    'pixelFlags_interpolatedCenter',
]

dia_objects.plot_coverage()

## 3. Fetch Gaia DR3 AGN candidates in the DP2 footprint

Use `gaiadr3.qso_candidates` (DSC-Combmod quasar probability > 0.9).  
Update the RA/Dec bounds below to match the DP2 sky coverage shown above.

In [ ]:
# ── Edit these bounds to match the DIA coverage map above ──────────────────
RA_MIN, RA_MAX   =   0,  360   # full sky first pass; refine after seeing coverage
DEC_MIN, DEC_MAX = -90,   15   # LSST primarily observes southern sky
QSO_PROB_MIN     =   0.9       # DSC-Combmod quasar probability threshold
# ───────────────────────────────────────────────────────────────────────────

Gaia.MAIN_GAIA_TABLE = 'gaiadr3.gaia_source'
Gaia.ROW_LIMIT = -1

adql = f"""
SELECT
    q.source_id,
    g.ra, g.dec,
    g.phot_g_mean_mag,
    g.phot_bp_mean_mag,
    g.phot_rp_mean_mag,
    g.parallax,
    g.parallax_over_error,
    q.classprob_dsc_combmod_quasar
FROM gaiadr3.gaia_source AS g
JOIN gaiadr3.qso_candidates AS q USING (source_id)
WHERE q.classprob_dsc_combmod_quasar > {QSO_PROB_MIN}
  AND g.dec BETWEEN {DEC_MIN} AND {DEC_MAX}
"""

print('Querying Gaia DR3 qso_candidates …')
job = Gaia.launch_job_async(adql)
gaia_agn_df = job.get_results().to_pandas()
print(f'Gaia AGN candidates fetched: {len(gaia_agn_df):,}')
gaia_agn_df.head(3)

In [ ]:
gaia_agn_df = gaia_agn_df.rename(columns={'source_id': 'gaia_source_id'})
gaia_agn_df = gaia_agn_df.dropna(subset=['ra', 'dec']).reset_index(drop=True)

gaia_cat = lsdb.from_dataframe(
    gaia_agn_df,
    ra_column='ra',
    dec_column='dec',
    catalog_name='gaia_dr3_agn',
)
print(f'LSDB Gaia AGN catalog: {len(gaia_cat):,} sources')

## 4. Cross-match Gaia AGN × Rubin DP2 DIA objects

In [ ]:
XMATCH_RADIUS_ARCSEC = 1.0

matched_cat = gaia_cat.crossmatch(
    dia_objects,
    radius_arcsec=XMATCH_RADIUS_ARCSEC,
    how='inner',
    suffixes=('_gaia', '_dia'),
)

print('Computing cross-match …')
matched_df = matched_cat.compute()
print(f'Matched AGN: {len(matched_df):,}')

## 5. Extract and quality-filter per-band lightcurves

For each matched DIA object we:
1. Unpack the nested DIA sources
2. Drop detections with any quality flag set
3. Split by band
4. Retain (object, band) pairs with baseline ≥ 30 days

In [ ]:
def extract_clean_lcs(row):
    """
    Return dict {band: (t, flux, flux_err)} for one DIA object row.
    Only returns bands with finite data; baseline filtering done outside.
    """
    sources = row.get(NESTED_COL, None)
    if sources is None or not isinstance(sources, pd.DataFrame) or sources.empty:
        return {}

    # ── flag removal ──────────────────────────────────────────────────────
    keep = pd.Series(True, index=sources.index)
    for fc in FLAG_COLS:
        if fc in sources.columns:
            keep &= ~sources[fc].fillna(False).astype(bool)

    # ── flux sanity ────────────────────────────────────────────────────────
    if FLUX_COL in sources.columns:
        keep &= sources[FLUX_COL].notna()
        keep &= sources[FLUXERR_COL].notna()
        keep &= sources[FLUXERR_COL] > 0
        keep &= np.isfinite(sources[FLUX_COL])
        keep &= np.isfinite(sources[FLUXERR_COL])

    sources = sources[keep]
    if sources.empty:
        return {}

    lcs = {}
    for band, grp in sources.groupby(BAND_COL):
        grp = grp.sort_values(TIME_COL)
        t    = grp[TIME_COL].values.astype(float)
        y    = grp[FLUX_COL].values.astype(float)
        yerr = grp[FLUXERR_COL].values.astype(float)
        if len(t) >= 5:   # minimum points needed for any fit
            lcs[band] = (t, y, yerr)
    return lcs


# Resolve column names after suffix-renaming from crossmatch
ra_col  = 'ra_dia'  if 'ra_dia'  in matched_df.columns else 'ra'
dec_col = 'dec_dia' if 'dec_dia' in matched_df.columns else 'dec'
oid_col = 'diaObjectId'

records = []
for _, row in matched_df.iterrows():
    obj_id   = row.get(oid_col, row.name)
    ra       = row.get(ra_col,  np.nan)
    dec      = row.get(dec_col, np.nan)
    gaia_id  = row.get('gaia_source_id', np.nan)
    g_mag    = row.get('phot_g_mean_mag', np.nan)
    qso_prob = row.get('classprob_dsc_combmod_quasar', np.nan)

    for band, (t, y, yerr) in extract_clean_lcs(row).items():
        baseline = float(t[-1] - t[0]) if len(t) > 1 else 0.0
        cadence  = len(t) / baseline if baseline > 0 else 0.0
        records.append(dict(
            diaObjectId=obj_id, gaia_source_id=gaia_id,
            ra=ra, dec=dec,
            g_mag=g_mag, qso_prob=qso_prob,
            band=band,
            t=t, y=y, yerr=yerr,
            n=len(t),
            baseline_days=baseline,
            cadence_per_day=cadence,
        ))

all_lc = pd.DataFrame(records)
print(f'Total (object, band) lightcurves before baseline cut: {len(all_lc):,}')

## 6. Select ≥ 200 lightcurves — baseline ≥ 30 days, densest cadence

In [ ]:
MIN_BASELINE_DAYS = 30
TARGET_N_LCS      = 200   # aim for at least this many; take all if fewer

qual = all_lc[all_lc['baseline_days'] >= MIN_BASELINE_DAYS].copy()
# Sort densest cadence first; take top TARGET_N_LCS if we have more
qual = qual.sort_values('cadence_per_day', ascending=False)
lc_df = qual.head(max(TARGET_N_LCS, len(qual))).reset_index(drop=True)

band_counts = lc_df.groupby('band')['diaObjectId'].count().rename('n_lcs')
print(f'Selected lightcurves: {len(lc_df):,}  (target: ≥{TARGET_N_LCS})')
print()
print('Per-band breakdown:')
print(band_counts.to_string())

## 7. Fit DRW model with EzTaoX

The DRW covariance is `k(Δt) = σ² exp(−|Δt|/τ)`.

Structure function: `SF(Δt) = σ √(2(1 − exp(−Δt/τ)))`

In [ ]:
def _call_eztaox(t, y, yerr):
    """
    Thin adapter so the rest of the notebook is independent of the exact
    eztaox API version.  Returns (tau_days, sigma_nJy, mean_flux, lnlike).
    Adjust the inner calls if your installed eztaox has a different interface.
    """
    # ── Try class-based API (eztaox >= 0.2 style) ────────────────────────
    try:
        model  = eztaox.DRW()
        result = model.fit(t, y, yerr)
        return float(result.tau), float(result.sigma), float(result.mean), float(result.lnlike)
    except AttributeError:
        pass

    # ── Try function-based API (eztaox 0.1 style) ────────────────────────
    try:
        result = eztaox.fit_drw(t, y, yerr)
        tau    = float(getattr(result, 'tau',    result[0]))
        sigma  = float(getattr(result, 'sigma',  result[1]))
        mean   = float(getattr(result, 'mean',   np.nanmean(y)))
        lnlike = float(getattr(result, 'lnlike', np.nan))
        return tau, sigma, mean, lnlike
    except AttributeError:
        pass

    # ── Fallback: direct CARMA(1,0) via eztaox.carma ────────────────────
    result = eztaox.carma_fit(t, y, yerr, p=1, q=0)
    tau    = float(result.tau)
    sigma  = float(result.sigma)
    return tau, sigma, float(np.nanmean(y)), float(getattr(result, 'lnlike', np.nan))


def fit_drw(t, y, yerr):
    """Fit DRW to one lightcurve. Returns dict of fit parameters."""
    try:
        tau, sigma, mean_flux, lnlike = _call_eztaox(t, y, yerr)
        return dict(tau=tau, sigma=sigma, mean_flux=mean_flux,
                    lnlike=lnlike, converged=True, error='')
    except Exception as exc:
        return dict(tau=np.nan, sigma=np.nan, mean_flux=float(np.nanmean(y)),
                    lnlike=np.nan, converged=False, error=str(exc))


print('Fitting DRW to', len(lc_df), 'lightcurves …')
fit_records = []
for idx, row in lc_df.iterrows():
    if idx % 50 == 0:
        print(f'  {idx}/{len(lc_df)}')
    res = fit_drw(row['t'], row['y'], row['yerr'])
    res.update({
        'diaObjectId':    row['diaObjectId'],
        'gaia_source_id': row['gaia_source_id'],
        'ra': row['ra'], 'dec': row['dec'],
        'g_mag': row['g_mag'],
        'qso_prob': row['qso_prob'],
        'band': row['band'],
        'n': row['n'],
        'baseline_days': row['baseline_days'],
        'cadence_per_day': row['cadence_per_day'],
    })
    fit_records.append(res)

drw_raw = pd.DataFrame(fit_records)
print(f'\nTotal fits:      {len(drw_raw):,}')
print(f'Converged:       {drw_raw["converged"].sum():,}')
print(f'Failed (logged): {(~drw_raw["converged"]).sum():,}')
if (~drw_raw['converged']).any():
    print('Sample errors:')
    print(drw_raw[~drw_raw['converged']]['error'].value_counts().head(3))

In [ ]:
drw = drw_raw[drw_raw['converged'] &
              (drw_raw['tau'] > 0) &
              np.isfinite(drw_raw['tau']) &
              np.isfinite(drw_raw['sigma'])].copy()

# Structure function at 10 days
drw['SF_10d'] = drw['sigma'] * np.sqrt(2.0 * (1.0 - np.exp(-10.0 / drw['tau'])))

# tau quality flag: well-constrained only if tau < baseline
drw['tau_constrained'] = drw['tau'] < drw['baseline_days']

print('Kept after quality cuts:', len(drw))
print(drw.groupby('band')[['tau', 'sigma', 'SF_10d']].describe().round(1))

## 8. Which band is most variable on ~10-day scales?

In [ ]:
sf10_by_band = (
    drw.groupby('band')['SF_10d']
    .agg(n='count', mean='mean', median='median', std='std')
    .reindex(BAND_ORDER)
    .dropna(subset=['mean'])
    .sort_values('median', ascending=False)
)

print('Median SF(10 day) per band [nJy]:')
print(sf10_by_band.to_string())
print()

most_variable_band = sf10_by_band.index[0]
print(f'>>> Most variable band at ~10-day scales: {most_variable_band}')
print(f'    Median SF(10d) = {sf10_by_band.loc[most_variable_band, "median"]:.2f} nJy')

# Bar chart
fig, ax = plt.subplots(figsize=(7, 4))
bands_plot = sf10_by_band.index.tolist()
medians    = sf10_by_band['median'].values
errs       = sf10_by_band['std'].values
colors     = [BAND_COLORS.get(b, 'gray') for b in bands_plot]
bars = ax.bar(bands_plot, medians, yerr=errs, color=colors,
               capsize=4, alpha=0.85, edgecolor='black', linewidth=0.6)
ax.set_xlabel('Band')
ax.set_ylabel('Median SF(10 d)  [nJy]')
ax.set_title('AGN Structure Function at 10-day Lag per Band\n(DRW fits, Rubin DP2 × Gaia DR3)')
ax.grid(axis='y', alpha=0.4)
ax.bar_label(bars, labels=[f'{v:.0f}' for v in medians], padding=3, fontsize=9)
plt.tight_layout()
plt.savefig('sf10_by_band.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Most variable AGN per band (bands with > 10 lightcurves)

In [ ]:
qualified_bands = sf10_by_band[sf10_by_band['n'] > 10].index.tolist()
print('Bands with >10 converged lightcurves:', qualified_bands)

top_per_band = {}  # band -> row from drw
for band in qualified_bands:
    sub = drw[drw['band'] == band]
    top_row = sub.loc[sub['SF_10d'].idxmax()]
    top_per_band[band] = top_row
    print(f'\n{band}-band most variable AGN')
    print(f'  diaObjectId      : {top_row["diaObjectId"]}')
    print(f'  Gaia source_id   : {int(top_row["gaia_source_id"]) if pd.notna(top_row["gaia_source_id"]) else "n/a"}')
    print(f'  RA, Dec          : {top_row["ra"]:.5f}, {top_row["dec"]:.5f}')
    print(f'  G mag            : {top_row["g_mag"]:.2f}')
    print(f'  DRW τ            : {top_row["tau"]:.1f} days')
    print(f'  DRW σ            : {top_row["sigma"]:.2f} nJy')
    print(f'  SF(10 d)         : {top_row["SF_10d"]:.2f} nJy')
    print(f'  N detections     : {int(top_row["n"])}')
    print(f'  Baseline         : {top_row["baseline_days"]:.0f} days')
    print(f'  τ constrained    : {top_row["tau_constrained"]}')

## 10. Lightcurve + DRW fit plots for each top AGN

In [ ]:
def drw_posterior(t_grid, t_obs, y_obs, yerr_obs, tau, sigma, mean_flux):
    """
    Analytic GP posterior mean and std for the DRW process on t_grid,
    conditioned on observations (t_obs, y_obs, yerr_obs).
    """
    def K(dt):
        return sigma**2 * np.exp(-np.abs(dt) / tau)

    n = len(t_obs)
    Knn = K(t_obs[:, None] - t_obs[None, :]) + np.diag(yerr_obs**2)
    Kmn = K(t_grid[:, None] - t_obs[None, :])   # (M, N)

    try:
        L     = np.linalg.cholesky(Knn + 1e-10 * np.eye(n))
        alpha = np.linalg.solve(L.T, np.linalg.solve(L, y_obs - mean_flux))
        mu    = mean_flux + Kmn @ alpha
        v     = np.linalg.solve(L, Kmn.T)        # (N, M)
        var   = sigma**2 - np.sum(v**2, axis=0)
        std   = np.sqrt(np.maximum(var, 0))
    except np.linalg.LinAlgError:
        mu  = np.full(len(t_grid), mean_flux)
        std = np.full(len(t_grid), sigma)

    return mu, std


n_bands = len(qualified_bands)
fig, axes = plt.subplots(n_bands, 1, figsize=(14, 4.5 * n_bands), squeeze=False)

for ax, band in zip(axes[:, 0], qualified_bands):
    top      = top_per_band[band]
    obj_id   = top['diaObjectId']
    lc_row   = lc_df[(lc_df['diaObjectId'] == obj_id) & (lc_df['band'] == band)].iloc[0]
    t        = lc_row['t']
    y        = lc_row['y']
    yerr     = lc_row['yerr']
    tau      = top['tau']
    sigma    = top['sigma']
    mf       = top['mean_flux']
    t0       = t.min()

    t_grid   = np.linspace(t.min(), t.max(), 600)
    mu, std  = drw_posterior(t_grid, t, y, yerr, tau, sigma, mf)

    col = BAND_COLORS.get(band, 'steelblue')
    ax.errorbar(t - t0, y, yerr=yerr, fmt='o', ms=3.5, alpha=0.65,
                color=col, ecolor='#888888', elinewidth=0.8, capsize=1.5,
                label='DIA psfFlux')
    ax.plot(t_grid - t0, mu, '-', color='#111111', lw=1.8, label='DRW posterior mean')
    ax.fill_between(t_grid - t0, mu - std, mu + std,
                    color='#111111', alpha=0.12, label='DRW ±1σ')
    ax.axhline(mf, color=col, lw=0.8, ls='--', alpha=0.6, label=f'Mean flux')

    constrained = '(τ constrained)' if top['tau_constrained'] else '(τ UNCONSTRAINED — treat fit with caution)'
    ax.set_title(
        f'{band}-band | diaObjectId={obj_id} | '
        f'τ={tau:.0f} d, σ={sigma:.0f} nJy, SF(10d)={top["SF_10d"]:.0f} nJy | '
        f'N={int(lc_row["n"])}, baseline={lc_row["baseline_days"]:.0f} d  {constrained}',
        fontsize=9.5
    )
    ax.set_xlabel('Days since first observation')
    ax.set_ylabel('psfFlux (nJy)')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.25)

plt.suptitle(
    'Most Variable AGN per Band — Rubin DP2 × Gaia DR3 — DRW fits via EzTaoX',
    fontsize=13, y=1.005
)
plt.tight_layout()
plt.savefig('top_agn_lightcurves.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Physical vs instrumental variability assessment

Key physical signatures of genuine AGN variability:
- **Bluer-is-brighter**: shorter wavelengths are more variable (accretion disk fluctuations)
- **τ ≈ 100–1000 days** (rest-frame): typical AGN damping timescales; very short τ may indicate PSF artifacts
- **Well-constrained τ**: τ should be smaller than the lightcurve baseline
- **Smooth, aperiodic variability**: no sharp single-epoch spikes

Instrumental/spurious flags:
- τ ≪ 10 days or τ ≫ baseline
- Variability not ordered by wavelength
- Spiky lightcurves from satellite trails, cosmic rays (should be caught by flags but not always)
- Very high σ relative to the mean flux (|σ/mean| ≫ 1 is suspicious)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# ── Panel A: τ distribution per band ──────────────────────────────────────
ax = axes[0, 0]
for band in BAND_ORDER:
    sub = drw[(drw['band'] == band) & (drw['tau'] > 0)]
    if len(sub) > 2:
        ax.hist(np.log10(sub['tau']), bins=25, alpha=0.55,
                label=band, color=BAND_COLORS.get(band, 'gray'), density=True)
ax.axvspan(np.log10(3), np.log10(30), alpha=0.07, color='red',
           label='< 30 d (suspect)')
ax.axvline(np.log10(100),  color='green', ls='--', lw=1, label='100 d')
ax.axvline(np.log10(1000), color='orange', ls='--', lw=1, label='1000 d')
ax.set_xlabel('log₁₀(τ / days)')
ax.set_ylabel('Density')
ax.set_title('DRW damping timescale τ distribution')
ax.legend(fontsize=7, ncol=2)
ax.grid(alpha=0.3)

# ── Panel B: Bluer-is-brighter test ───────────────────────────────────────
ax = axes[0, 1]
bands_avail = [b for b in BAND_ORDER if b in sf10_by_band.index]
wl   = [BAND_WL_UM[b] for b in bands_avail]
med  = [sf10_by_band.loc[b, 'median'] for b in bands_avail]
ax.plot(wl, med, 'o-', color='steelblue', ms=9, lw=2)
for b, w, m in zip(bands_avail, wl, med):
    ax.annotate(b, (w, m), textcoords='offset points',
                xytext=(0, 7), ha='center', fontsize=11, color=BAND_COLORS.get(b))
# Bluer-is-brighter = SF decreases monotonically with wavelength (left → right)
ax.set_xlabel('Effective wavelength (μm)')
ax.set_ylabel('Median SF(10 d) [nJy]')
ax.set_title('Bluer-is-brighter test\n(↗ physical AGN; flat/↘ instrumental?)')
ax.invert_xaxis()
ax.grid(alpha=0.3)

# Annotate whether the trend is present
from scipy.stats import spearmanr
if len(wl) >= 3:
    rho, pval = spearmanr(wl, med)
    verdict = 'bluer-is-brighter ✓' if rho < -0.5 and pval < 0.1 else 'trend unclear'
    ax.text(0.05, 0.95, f'Spearman ρ={rho:.2f}, p={pval:.2f}\n{verdict}',
            transform=ax.transAxes, va='top', fontsize=9,
            bbox=dict(boxstyle='round', fc='white', alpha=0.7))

# ── Panel C: τ / baseline (constraint quality) ────────────────────────────
ax = axes[1, 0]
for band in BAND_ORDER:
    sub = drw[drw['band'] == band]
    if len(sub) > 2:
        ratio = sub['tau'] / sub['baseline_days']
        ax.hist(ratio.clip(0, 5), bins=25, alpha=0.55,
                label=band, color=BAND_COLORS.get(band, 'gray'), density=True)
ax.axvline(1.0, color='black', ls='--', lw=1.5, label='τ = baseline')
ax.set_xlabel('τ / baseline')
ax.set_ylabel('Density')
ax.set_title('DRW constraint quality\n(τ > baseline → timescale unconstrained)')
ax.legend(fontsize=7, ncol=2)
ax.grid(alpha=0.3)
frac_constrained = drw['tau_constrained'].mean()
ax.text(0.98, 0.95, f'τ constrained: {frac_constrained:.0%}',
        transform=ax.transAxes, ha='right', va='top', fontsize=9,
        bbox=dict(boxstyle='round', fc='white', alpha=0.7))

# ── Panel D: σ/|mean_flux| — fractional variability amplitude ─────────────
ax = axes[1, 1]
drw['frac_var'] = drw['sigma'] / np.abs(drw['mean_flux']).clip(lower=1.0)
for band in BAND_ORDER:
    sub = drw[(drw['band'] == band) & np.isfinite(drw['frac_var'])]
    if len(sub) > 2:
        ax.hist(np.log10(sub['frac_var'].clip(1e-4, 1e2)), bins=25, alpha=0.55,
                label=band, color=BAND_COLORS.get(band, 'gray'), density=True)
ax.axvline(np.log10(0.1), color='green', ls='--', lw=1, label='10% variability')
ax.axvline(np.log10(1.0), color='red',   ls='--', lw=1, label='100% (suspect)')
ax.set_xlabel('log₁₀(σ / |mean flux|)')
ax.set_ylabel('Density')
ax.set_title('Fractional variability amplitude σ/|<flux>|\n(> 100% often instrumental)')
ax.legend(fontsize=7, ncol=2)
ax.grid(alpha=0.3)

plt.suptitle('Physical vs Instrumental Variability Diagnostics', fontsize=13)
plt.tight_layout()
plt.savefig('drw_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Summary

In [ ]:
print('=' * 65)
print('SUMMARY')
print('=' * 65)
print(f'  Cross-matched Gaia AGN (P_qso>0.9) × Rubin DP2 DIA objects')
print(f'  Match radius         : {XMATCH_RADIUS_ARCSEC} arcsec')
print(f'  Matched sources      : {len(matched_df):,}')
print(f'  Lightcurves selected : {len(lc_df):,}  (≥{MIN_BASELINE_DAYS} d baseline)')
print(f'  DRW fits converged   : {len(drw):,}')
print()
print(f'  Most variable band at ~10-day scales: {most_variable_band}')
print(f'  (Median SF(10d) = {sf10_by_band.loc[most_variable_band, "median"]:.2f} nJy)')
print()
print('  Most variable AGN per qualified band (SF at 10 days):')
for band in qualified_bands:
    row = top_per_band[band]
    print(f'    {band}: diaObjectId={row["diaObjectId"]}  '
          f'SF(10d)={row["SF_10d"]:.1f} nJy  '
          f'τ={row["tau"]:.0f}d  '
          f'constrained={row["tau_constrained"]}')
print()
print('  Variability assessment:')
print(f'    τ-constrained fraction : {drw["tau_constrained"].mean():.0%}')
if len(wl) >= 3:
    rho, pval = spearmanr(wl, med)
    physical = rho < -0.5 and pval < 0.1
    print(f'    Bluer-is-brighter      : {"YES" if physical else "UNCERTAIN"}')
    print(f'    (Spearman ρ={rho:.2f}, p={pval:.2f})')
median_frac = drw['frac_var'].median()
print(f'    Median σ/|mean flux|   : {median_frac:.2f}')
print(f'    Fraction σ/|mean|>1    : {(drw["frac_var"]>1).mean():.0%}  '
      f'(high values → possible instrumental contribution)')
print()
print('  Interpretation:')
print('    - If bluer-is-brighter holds and τ ≈ 100–1000 days, variability')
print('      is consistent with physical accretion-disk fluctuations.')
print('    - Sources with τ ≪ 30 days or τ > baseline are DRW-unconstrained;')
print('      their SF(10d) should not be over-interpreted.')
print('    - Very short commissioning baselines (<90 days) make τ estimates')
print('      unreliable for typical AGN (τ_AGN >> baseline); treat as lower')
print('      bounds on τ and flag for re-analysis with Year-1 data.')
print('=' * 65)